# Preprocessing

In [54]:
import pandas as pd
import numpy as np

data = pd.read_parquet('../data/processed/data_with_listeners.parquet')
data = data.dropna().reset_index(drop=True)
orig_data= pd.read_parquet('../data/processed/orig_data.parquet')
target = 'popularity'

In [55]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, dtype=int)
encoded_columns = encoder.fit_transform(data[['track_genre']])

encoded_data = pd.DataFrame(encoded_columns, columns=encoder.get_feature_names_out(['track_genre']), index=data.index)
data_copy = data.copy()

data = pd.concat([data_copy.drop(columns=['track_genre']), encoded_data], axis=1)
data.reset_index(drop=True)
data = data.drop(columns='artist_fame_loo')

In [56]:
data.describe()

,popularity,duration_ms,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,...,track_genre_spanish,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music
count,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,...,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000,66804.000000
mean,38.776376,12.267978,0.561721,0.640067,5.294354,-8.497902,0.633420,0.091079,0.327851,0.176532,...,0.011990,0.010658,0.009775,0.008532,0.012903,0.007095,0.009969,0.009865,0.013278,0.011886
std,17.727542,0.359194,0.174580,0.254948,3.554427,5.207748,0.481874,0.121272,0.336407,0.325992,...,0.108843,0.102687,0.098384,0.091977,0.112859,0.083935,0.099349,0.098831,0.114462,0.108372
min,0.000000,11.002266,0.051300,0.000020,0.000000,-46.591000,0.000000,0.022100,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,24.000000,12.065445,0.450000,0.465000,2.000000,-10.288000,0.000000,0.036100,0.016200,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,39.000000,12.273093,0.575000,0.682000,5.000000,-7.209500,1.000000,0.049500,0.194000,0.000069,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,52.000000,12.485730,0.690000,0.857000,8.000000,-5.133750,1.000000,0.088900,0.621000,0.108000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,100.000000,13.304685,0.984000,1.000000,11.000000,4.532000,1.000000,0.963000,0.996000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


# Train Linear Regression Model (Oridinary Least Squares)

In [57]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

X = data.copy()
y = data_copy[target]
X = X.drop(columns=target)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [58]:
# Cross-validation on the traning set

pipe_lin_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LinearRegression())
])

score_r2 = cross_val_score(pipe_lin_reg, X_train, y_train, cv=10, scoring='r2')
score_mae = cross_val_score(pipe_lin_reg, X_train, y_train, cv=10, scoring='neg_mean_absolute_error')
score_rmse = cross_val_score(pipe_lin_reg, X_train, y_train, cv=10, scoring='neg_root_mean_squared_error')

print("Plain: R2=", score_r2.mean())
print("Plain: MAE=", -score_mae.mean())
print("Plain: RMSE=", -score_rmse.mean())

Plain: R2= 0.5978930317526151
Plain: MAE= 8.276765747423964
Plain: RMSE= 11.220437161249027


In [59]:
# Now we can actually train the model
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

pipe_lin_reg.fit(X_train, y_train)

y_train_pred = pipe_lin_reg.predict(X_train)
y_test_pred = pipe_lin_reg.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))

print("Train MAE:", mean_absolute_error(y_train, y_train_pred))
print("Test MAE:", mean_absolute_error(y_test, y_test_pred))

print("Train RMSE:", root_mean_squared_error(y_train, y_train_pred))
print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred))


Train R2: 0.600851306358529
Test R2: 0.5983075798744264
Train MAE: 8.25245448417578
Test MAE: 8.26888608140734
Train RMSE: 11.181045836366748
Test RMSE: 11.309977016968473


# Train Lasso Regressor

In [60]:
from sklearn.linear_model import Lasso
from sklearn.model_selection import GridSearchCV, cross_validate, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
import numpy as np

alphas = np.logspace(-3, 1, 5)

pipe_lasso = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Lasso(max_iter=5000, random_state=667, tol=1e-3))
])

param_grid = {
    "model__alpha": alphas
}

inner_cv = KFold(n_splits=5, shuffle=True, random_state=667)
outer_cv = KFold(n_splits=10, shuffle=True, random_state=668)

search = GridSearchCV(
    estimator=pipe_lasso,
    param_grid=param_grid,
    cv=inner_cv,
    scoring="r2",
    n_jobs=-1
)

cv_results = cross_validate(
    estimator=search,
    X=X,
    y=y,
    cv=outer_cv,
    scoring={
        "r2": "r2",
        "mae": "neg_mean_absolute_error",
        "rmse": "neg_root_mean_squared_error",
    },
    n_jobs=-1,
    return_estimator=True
)

print("Nested CV R2:", cv_results["test_r2"].mean())
print("Nested CV MAE:", -cv_results["test_mae"].mean())
print("Nested CV RMSE:", -cv_results["test_rmse"].mean())

outer_best_alphas = [
    estimator.best_params_["model__alpha"]
    for estimator in cv_results["estimator"]
]

print("Best alpha per outer fold:", outer_best_alphas)
print("Median best alpha:", np.median(outer_best_alphas))

Nested CV R2: 0.5983513394650227
Nested CV MAE: 8.276601252583458
Nested CV RMSE: 11.23331729349871
Best alpha per outer fold: [np.float64(0.001), np.float64(0.001), np.float64(0.001), np.float64(0.001), np.float64(0.01), np.float64(0.001), np.float64(0.001), np.float64(0.001), np.float64(0.001), np.float64(0.001)]
Median best alpha: 0.001


# Now we train MLP

In [61]:
import pandas as pd
import numpy as np

from sklearn.model_selection import GroupShuffleSplit, GridSearchCV, KFold
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

df = pd.read_parquet("../data/processed/orig_data_with_listeners.parquet")

target = "popularity"
group_col = "primary_artist"

numeric_features = [
    "duration_ms",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature",
    "artists_listeners",
]

categorical_features = [
    "explicit",
    "track_genre",
]

needed_cols = [target, group_col] + numeric_features + categorical_features
df = df[needed_cols].dropna()

X = df[numeric_features + categorical_features]
y = df[target]
groups = df[group_col]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42,
)

train_idx, test_idx = next(splitter.split(X, y, groups=groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [62]:
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_features),
])

mlp = MLPRegressor(
    hidden_layer_sizes=(128, 64),
    activation="relu",
    solver="adam",
    alpha=1e-4,
    learning_rate_init=1e-3,
    max_iter=500,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=20,
    random_state=42,
)

pipe_mlp = Pipeline([
    ("preprocess", preprocess),
    ("model", mlp),
])

In [63]:
pipe_mlp.fit(X_train, y_train)

y_train_pred = pipe_mlp.predict(X_train)
y_test_pred = pipe_mlp.predict(X_test)

print("Train R2:", r2_score(y_train, y_train_pred))
print("Test R2:", r2_score(y_test, y_test_pred))

print("Train MAE:", mean_absolute_error(y_train, y_train_pred))
print("Test MAE:", mean_absolute_error(y_test, y_test_pred))

print("Train RMSE:", root_mean_squared_error(y_train, y_train_pred))
print("Test RMSE:", root_mean_squared_error(y_test, y_test_pred))

Train R2: 0.7284288049849075
Test R2: 0.6494341734710606
Train MAE: 6.409218320956863
Test MAE: 7.284589395368594
Train RMSE: 9.22853708139005
Test RMSE: 10.539666082292202


In [64]:
data

,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,...,track_genre_spanish,track_genre_study,track_genre_swedish,track_genre_synth-pop,track_genre_tango,track_genre_techno,track_genre_trance,track_genre_trip-hop,track_genre_turkish,track_genre_world-music
0,100,11.963644,False,0.714,0.472,2,-7.375,1,0.0864,0.01300,...,0,0,0,0,0,0,0,0,0,0
1,99,12.200748,False,0.621,0.782,2,-5.548,1,0.0440,0.01250,...,0,0,0,0,0,0,0,0,0,0
2,98,11.999282,False,0.835,0.679,7,-5.329,0,0.0364,0.58300,...,0,0,0,0,0,0,0,0,0,0
3,98,12.073906,True,0.561,0.965,7,-3.673,0,0.0343,0.00383,...,0,0,0,0,0,0,0,0,0,0
4,97,12.092725,True,0.911,0.712,1,-5.105,0,0.0817,0.09010,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
66799,0,12.431350,False,0.632,0.961,2,-3.576,0,0.0407,0.26900,...,0,0,0,0,0,0,0,0,0,0
66800,0,12.716699,False,0.803,0.838,0,-2.450,1,0.0949,0.59100,...,0,0,0,0,0,0,0,0,0,0
66801,0,12.223524,False,0.841,0.853,2,-2.131,1,0.0518,0.42400,...,0,0,0,0,0,0,0,0,0,0
66802,0,12.657744,False,0.667,0.733,9,-5.703,0,0.0427,0.75100,...,0,0,0,0,0,0,0,0,0,0
